# Data Preparation

Notebook unico che genera il file dati **`data/prepared_data.pt`** consumato da:

- gli script di tuning in `experiments/optuna_variant*.py`
- i notebook in `variants/variant*.ipynb`

Il codice di generazione è estratto da `main.ipynb` (celle 0-16): albero di processo
&rarr; rete di Petri &rarr; generazione tracce &rarr; clustering/bilanciamento &rarr;
codifica regioni/task/completo &rarr; assegnazione tempi &rarr; tensori + dizionari.

> **Nota sui tempi.** In `main.ipynb` `data_times` veniva creato con `dtype=torch.long`,
> ma i tempi sono float normalizzati in `[0, 1]`: con `long` verrebbero azzerati. Qui
> usiamo `torch.float32`, coerente con come i notebook delle varianti li consumano
> (`.float().mean()`) e con la proiezione interna del `TimeTransformer`.


## 1. Import e settings

In [2]:
import os
import random

import numpy as np
import pandas as pd
import torch
from sklearn import tree as sktree
from sklearn.cluster import DBSCAN
from collections import Counter

from core import (
    PetriNetP, Generator,
    SEED_STRING, PARSER, replace_underscores, replace_random_underscore,
    TracePatternMiner,
)
from utils import (
    createNAryTree,
    get_encoding, get_decoding,
    create_loop_data, create_task_data,
    getall_traces, create_distance_matrix, get_balance_traces_by_cluster,
    assign_time,
)

# --- SETTINGS ---
NARY = 1
PROBABILITIES = 0.33, 0.33, 0.34, 0
OUTPUT_PATH = "data/prepared_data.pt"

# Seed per rendere riproducibile la preparazione dei dati (generazione tracce,
# clustering e assegnazione tempi usano randomness). main.ipynb non lo fa.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 2. Albero di processo

Si usa l'albero giocattolo "2 LOOP" attivo in `main.ipynb` (deterministico).

In [3]:
from lark import Tree, Token

# Albero giocattolo "2 LOOP" (quello attivo in main.ipynb).
tree = Tree('xor', [
    Tree('sequential', [
        Tree('loop', [
            Tree('task', [Token('NAME', 'T1')])
        ]),
        Tree('xor', [
            Tree('task', [Token('NAME', 'T2')]),
            Tree('task', [Token('NAME', 'T3')])
        ])
    ]),
    Tree('parallel', [
        Tree('loop', [   # loop inserito in un ramo del parallel
            Tree('task', [Token('NAME', 'T4')])
        ]),
        Tree('task', [Token('NAME', 'T5')])
    ])
])

iterations = 15  # Quante iterazioni diverse (numero regioni prima di minimizzare in teoria)
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

if NARY:  # Se true allora faccio l'albero ennario (semplifico)
    tree = createNAryTree(tree)

print(tree)

Tree('xor', [Tree('sequential', [Tree('parallel', [Tree('xor', [Tree('sequential', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])]), Tree('sequential', [Tree('task', [Token('NAME', 'T3')]), Tree('task', [Token('NAME', 'T4')]), Tree('task', [Token('NAME', 'T5')])]), Tree('task', [Token('NAME', 'T6')]), Tree('task', [Token('NAME', 'T7')])]), Tree('sequential', [Tree('xor', [Tree('task', [Token('NAME', 'T8')]), Tree('task', [Token('NAME', 'T9')])]), Tree('task', [Token('NAME', 'T10')])])]), Tree('xor', [Tree('task', [Token('NAME', 'T11')]), Tree('task', [Token('NAME', 'T12')])])]), Tree('parallel', [Tree('task', [Token('NAME', 'T13')]), Tree('task', [Token('NAME', 'T14')])]), Tree('parallel', [Tree('task', [Token('NAME', 'T15')]), Tree('task', [Token('NAME', 'T16')])])])


## 3. Rete di Petri e generazione tracce

In [4]:
net = PetriNetP(tree)

# Oggetto Generator
generator = Generator(50000, net)
generator.generateTrace(False)  # con False non puliamo la traccia
print(f"Tracce generate: {len(generator.generatedTraces)}")

Tracce generate: 50000


## 4. Decision tree per i loop

Usati per dare un "senso" ai loop, poi le tracce vengono rigenerate condizionatamente.

In [5]:
classifier_dict_loop = {}

for loop in net.loop_regions:
    result = create_loop_data(loop, generator.generatedTraces)
    if result is None:
        continue

    x, y, dict_loop_step_encoding, max_len = result

    # class_weight='balanced' equivale ad aver pompato le tracce
    decisiontree = sktree.DecisionTreeClassifier(max_depth=5, class_weight='balanced')
    decisiontree = decisiontree.fit(x, y)

    classifier_dict_loop[loop] = (decisiontree, dict_loop_step_encoding, max_len)

# Rigenero le tracce utilizzando i classificatori
generator.generateTraceCond(classifier_dict_loop)
print(f"Loop classifiers: {len(classifier_dict_loop)}")

Loop classifiers: 0


## 5. Codifica tracce (regioni + task)

In [6]:
# Codifica delle tracce generate
traceEncoded_regions, traceEncoded_tasks = get_encoding(
    generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses
)

# Numero regioni e numero task effettivo
num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

# Dataframe unico (regioni + task)
df_traces_complete = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(f"num_regions={num_regions}, num_tasks={num_tasks}")

num_regions=11, num_tasks=16


## 6. Matrice di distanza e clustering (DBSCAN)

In [7]:
# Tutte le tracce -> matrice di distanza (weighted levenshtein con hamming)
all_traces = getall_traces(num_regions + num_tasks, df_traces_complete)

trace_counts = Counter(all_traces)
print(f"Tracce uniche: {len(trace_counts)}")
traces_encoded = list(trace_counts.keys())
trace_weights = list(trace_counts.values())
n_traces = len(traces_encoded)

distance_matrix = create_distance_matrix(n_traces, traces_encoded, num_regions + num_tasks)

# Clusterizzo
dbscan = DBSCAN(eps=6, min_samples=1, metric="precomputed")
dbscan.fit(distance_matrix, sample_weight=trace_weights)
cluster_labels = dbscan.labels_
print(f"Clusters trovati: {len(set(cluster_labels))}")

Tracce uniche: 1152
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
27

## 7. Bilanciamento per cluster

In [16]:
# Tot tracce per cluster -> bilancio per non avere prevalentemente tracce dello stesso cluster
num_trace_per_cluster = 5000
balanced_traces, balanced_columns = get_balance_traces_by_cluster(
    traces_encoded, cluster_labels, num_trace_per_cluster
)

df_traces_balanced = pd.DataFrame(balanced_columns).T  # Trasposta
df_traces_balanced.index = df_traces_complete.index
print(f"Steps bilanciati: {df_traces_balanced.shape[1]}")

Steps bilanciati: 190000


## 8. Funzioni di codifica/decodifica (completo, regioni, task)

In [17]:
# --- REGIONE + TASK (completo) ---
df_regions = df_traces_balanced.head(num_regions).copy()
df_tasks = df_traces_balanced.tail(num_tasks).copy()

df_traces_balanced = df_traces_balanced.T.copy()

unique_columns_complete = df_traces_balanced.drop_duplicates()
unique_tuple_complete = [tuple(x) for x in unique_columns_complete.values]

bit_to_id_traces_complete = {v: i for i, v in enumerate(unique_tuple_complete)}
id_to_bit_traces_complete = {i: v for i, v in enumerate(unique_tuple_complete)}
vocab_size_complete = len(unique_columns_complete)

encode_complete = lambda a: [bit_to_id_traces_complete[tuple(x)] for x in a]
decode_complete = lambda b: [id_to_bit_traces_complete[x] for x in b]

# --- SOLO REGIONE ---
df_regions = df_regions.T
unique_columns_regions = df_regions.drop_duplicates()
unique_tuple_regions = [tuple(x) for x in unique_columns_regions.values]

bit_to_id_traces_regions = {v: i for i, v in enumerate(unique_tuple_regions)}
id_to_bit_traces_regions = {i: v for i, v in enumerate(unique_tuple_regions)}
vocab_size_regions = len(unique_columns_regions)

encode_regions = lambda a: [bit_to_id_traces_regions[tuple(x)] for x in a]
decode_regions = lambda b: [id_to_bit_traces_regions[x] for x in b]

# --- SOLO TASK ---
df_tasks = df_tasks.T
unique_columns_tasks = df_tasks.drop_duplicates()
unique_tuple_tasks = [tuple(x) for x in unique_columns_tasks.values]

bit_to_id_traces_tasks = {v: i for i, v in enumerate(unique_tuple_tasks)}
id_to_bit_traces_tasks = {i: v for i, v in enumerate(unique_tuple_tasks)}
vocab_size_tasks = len(unique_columns_tasks)

encode_tasks = lambda a: [bit_to_id_traces_tasks[tuple(x)] for x in a]
decode_tasks = lambda b: [id_to_bit_traces_tasks[x] for x in b]

print(f"vocab_complete={vocab_size_complete}, vocab_regions={vocab_size_regions}, vocab_tasks={vocab_size_tasks}")

vocab_complete=62, vocab_regions=17, vocab_tasks=40


## 9. Pattern temporali per task

In [18]:
# Decodifico le tracce bilanciate
traces_balanced_decoded = get_decoding(balanced_traces, net.regions, net.tasks)

possible_tasks = []  # per ogni task consideriamo sia start_'task' sia end_'task'
for task in net.tasks:
    possible_tasks.append(f"start_{task}")
    possible_tasks.append(f"end_{task}")

k_partition = 3
max_depth = 5

# Seleziono k-pattern per differenziare i tempi delta dei vari step
tree_times_task_dict = {}
for task in possible_tasks:
    miner = TracePatternMiner()
    miner.root.name = task  # la radice e' la parte di task che sto usando

    num_traces_for_task = 0
    for trace in traces_balanced_decoded:
        if task in trace:
            idx = trace.index(task)
            subtrace_task = trace[:idx]              # traccia fino al task (escluso)
            reversed_subtrace = list(reversed(subtrace_task))
            miner.fit_trace(reversed_subtrace, max_depth)
            num_traces_for_task += 1

    if num_traces_for_task == 0 or len(miner.nodes) == 0:
        tree_times_task_dict[task] = {
            "partitions": {"RESIDUALS": {"numTraces": num_traces_for_task, "nodes": []}},
            "times_map": {"RESIDUALS": 0.0},
        }
        continue

    partitions = miner.select_patterns(num_traces_for_task, k_partition)
    times_map = miner.create_normalized_time_map(partitions)
    tree_times_task_dict[task] = {"partitions": partitions, "times_map": times_map}

print(f"Pattern temporali per {len(tree_times_task_dict)} task-step.")

{'RESIDUALS': {'numTraces': 5000, 'nodes': [<core.models.PatternTree.NodoPattern object at 0x781943197bb0>, <core.models.PatternTree.NodoPattern object at 0x781943197a30>, <core.models.PatternTree.NodoPattern object at 0x781943197640>, <core.models.PatternTree.NodoPattern object at 0x7819431976a0>, <core.models.PatternTree.NodoPattern object at 0x781943197580>, <core.models.PatternTree.NodoPattern object at 0x781943197610>, <core.models.PatternTree.NodoPattern object at 0x781943197a00>, <core.models.PatternTree.NodoPattern object at 0x781943197460>, <core.models.PatternTree.NodoPattern object at 0x781943197550>, <core.models.PatternTree.NodoPattern object at 0x781943197400>, <core.models.PatternTree.NodoPattern object at 0x7819431973d0>, <core.models.PatternTree.NodoPattern object at 0x781943197430>, <core.models.PatternTree.NodoPattern object at 0x7819431973a0>, <core.models.PatternTree.NodoPattern object at 0x781943197340>, <core.models.PatternTree.NodoPattern object at 0x78194319731

## 10. Assegnazione tempi alle tracce

In [19]:
times = []
trace_times_list = []
window = 5
for trace in traces_balanced_decoded:
    this_trace_times = []
    for i, element in enumerate(trace):
        if i == 0:
            this_trace_times.append(0)
            times.append(0)
            continue

        this_current_trace = trace[:i]
        times_map = tree_times_task_dict[element]["times_map"]

        generated_time = assign_time(this_current_trace, times_map, window)
        this_trace_times.append(generated_time)
        times.append(generated_time)

    trace_times_list.append(this_trace_times)

print(f"Tempi assegnati: {len(times)} step.")

Tempi assegnati: 190000 step.


## 11. Regression tree per i tempi dei task + encoding degli step

In [20]:
all_unique_steps = set(element for trace in traces_balanced_decoded for element in trace)
all_unique_steps.add("PAD")  # carattere speciale
dict_task_step_encoding = {task: i for i, task in enumerate(all_unique_steps)}

classifier_dict_tasks = {}
for task in possible_tasks:
    result = create_task_data(task, traces_balanced_decoded, trace_times_list, dict_task_step_encoding)
    if result is None:
        continue

    x, y, max_len = result
    regression_tree = sktree.DecisionTreeRegressor(max_depth=5)
    regression_tree = regression_tree.fit(x, y)
    classifier_dict_tasks[task] = (regression_tree, max_len)

print(f"Regression tree per {len(classifier_dict_tasks)} task-step.")

Regression tree per 32 task-step.


## 12. Tensori e split train/val

In [21]:
# Tensori con le colonne codificate in interi
data_complete = torch.tensor(encode_complete(df_traces_balanced.values), dtype=torch.long)
data_regions = torch.tensor(encode_regions(df_regions.values), dtype=torch.long)
data_tasks = torch.tensor(encode_tasks(df_tasks.values), dtype=torch.long)
# NB: float32 (non long): i tempi sono normalizzati in [0,1] - vedi nota in testa.
data_times = torch.tensor(times, dtype=torch.float32)

n = int(0.7 * len(df_traces_balanced))
print(f"data_complete={tuple(data_complete.shape)}, data_times={tuple(data_times.shape)}, n={n}")

data_complete=(190000,), data_times=(190000,), n=133000


## 13. Salvataggio `data/prepared_data.pt`

Dizionario con le 19 chiavi attese da varianti e script optuna.

In [22]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

info = {
    # split
    'n': n,
    # tensori
    'data_complete': data_complete,
    'data_regions': data_regions,
    'data_tasks': data_tasks,
    'data_times': data_times,
    # vocab sizes
    'vocab_size_complete': vocab_size_complete,
    'vocab_size_regions': vocab_size_regions,
    'vocab_size_tasks': vocab_size_tasks,
    # dimensioni
    'num_regions': num_regions,
    'num_tasks': num_tasks,
    # codifica/decodifica completo
    'bit_to_id_complete': bit_to_id_traces_complete,
    'id_to_bit_complete': id_to_bit_traces_complete,
    # codifica/decodifica regioni
    'bit_to_id_regions': bit_to_id_traces_regions,
    'id_to_bit_regions': id_to_bit_traces_regions,
    # codifica/decodifica task
    'bit_to_id_tasks': bit_to_id_traces_tasks,
    'id_to_bit_tasks': id_to_bit_traces_tasks,
    # oggetti per generazione / valutazione
    'net': net,
    'classifier_dict_tasks': classifier_dict_tasks,
    'dict_task_step_encoding': dict_task_step_encoding,
}

torch.save(info, OUTPUT_PATH)
print(f"Salvato {OUTPUT_PATH} con {len(info)} chiavi:")
print(sorted(info.keys()))

Salvato data/prepared_data.pt con 19 chiavi:
['bit_to_id_complete', 'bit_to_id_regions', 'bit_to_id_tasks', 'classifier_dict_tasks', 'data_complete', 'data_regions', 'data_tasks', 'data_times', 'dict_task_step_encoding', 'id_to_bit_complete', 'id_to_bit_regions', 'id_to_bit_tasks', 'n', 'net', 'num_regions', 'num_tasks', 'vocab_size_complete', 'vocab_size_regions', 'vocab_size_tasks']


## 14. Verifica (ricarico e controllo le chiavi attese)

In [23]:
EXPECTED_KEYS = {
    'n', 'data_complete', 'data_regions', 'data_tasks', 'data_times',
    'vocab_size_complete', 'vocab_size_regions', 'vocab_size_tasks',
    'num_regions', 'num_tasks',
    'bit_to_id_complete', 'id_to_bit_complete',
    'bit_to_id_regions', 'id_to_bit_regions',
    'bit_to_id_tasks', 'id_to_bit_tasks',
    'net', 'classifier_dict_tasks', 'dict_task_step_encoding',
}

reloaded = torch.load(OUTPUT_PATH, map_location='cpu', weights_only=False)
missing = EXPECTED_KEYS - set(reloaded.keys())
assert not missing, f"Chiavi mancanti: {missing}"
print("OK - tutte le chiavi attese sono presenti.")
print("data_times dtype:", reloaded['data_times'].dtype, "| mean:", round(reloaded['data_times'].float().mean().item(), 4))

OK - tutte le chiavi attese sono presenti.
data_times dtype: torch.float32 | mean: 0.3745
